# 在 PyTorch 中使用 AMD RX 5700 XT

[English](install_pytorch_mps_rx5700xt.ipynb) | [简体中文](install_pytorch_mps_rx5700xt_CN.ipynb)

可以用。在这台 Ryzen 9 3900X 黑苹果上，PyTorch 通过 Apple 的
**MPS/Metal** 后端调用 RX 5700 XT，并把它显示为 `mps:0`。

整套流程只有三步：

1. 安装已经验证过的 Python 和 PyTorch 版本；
2. 确认系统能够找到 `mps:0`；
3. 跑一个简短的矩阵测试，确认显卡确实在工作。

macOS 下不需要安装 CUDA 或 ROCm。

## 安装前先确认

这份教程适用于 **x86_64 黑苹果**，不是 Apple Silicon。实测配置如下：

| 项目 | 配置 |
|---|---|
| CPU / GPU | Ryzen 9 3900X / Radeon RX 5700 XT 8 GB |
| macOS | Sequoia 15.2，x86_64，Metal 3 |
| Python | 3.11 |
| PyTorch | 2.2.2 |
| torchvision / torchaudio | 0.17.2 / 2.2.2 |
| NumPy | 1.26.4 |

PyTorch 2.2.x 是最后一代提供官方 macOS x86_64 安装包的版本，所以这里使用的
版本看起来较旧。请单独创建环境，不要改动系统 Python。

先下载仓库并进入教程目录：

```bash
git clone https://github.com/hermanzhaozzzz/My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh.git
cd My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh/tutorial_pytorch-via-this_AMD5700xtGPU
```

下面的 **`uv` 和 Conda 任选一种**，不要在同一个环境中混用。

参考：[PyTorch 2.2 发布说明](https://pytorch.org/blog/pytorch2-2/) ·
[PyTorch MPS 文档](https://docs.pytorch.org/docs/stable/notes/mps.html)

## 使用 `uv` 安装（推荐）

如果还没有安装 `uv`：

```bash
brew install uv
```

创建环境并安装已经验证过的版本：

```bash
uv python install 3.11
uv venv --python 3.11 .venv

uv pip install --python .venv/bin/python \
  "numpy==1.26.4" \
  "torch==2.2.2" \
  "torchvision==0.17.2" \
  "torchaudio==2.2.2" \
  "jupyterlab>=4,<5" \
  "ipykernel>=6,<8"
```

用刚创建的环境启动 JupyterLab：

```bash
env -u PYTORCH_ENABLE_MPS_FALLBACK .venv/bin/jupyter lab
```

打开 `install_pytorch_mps_rx5700xt_CN.ipynb`。直接调用 `.venv/bin/jupyter`
可以避免 Jupyter 悄悄切换到电脑上的其他 Python 环境。

## 或者使用 Conda 安装

如果你平时使用 Anaconda 或 Miniconda，可以改用下面的方法：

```bash
conda create -n pytorch-mps-rx5700xt python=3.11 pip -y
conda activate pytorch-mps-rx5700xt

python -m pip install \
  "numpy==1.26.4" \
  "torch==2.2.2" \
  "torchvision==0.17.2" \
  "torchaudio==2.2.2" \
  "jupyterlab>=4,<5" \
  "ipykernel>=6,<8"
```

然后启动 JupyterLab：

```bash
env -u PYTORCH_ENABLE_MPS_FALLBACK jupyter lab
```

不要把这个环境升级到 PyTorch 2.3 或更高版本；这些版本没有这台电脑需要的
macOS x86_64 安装包。

## 第一步：检查环境

在 JupyterLab 中选择 **Kernel → Restart Kernel and Run All Cells**。下面的
代码会在性能测试前检查当前环境。

正确结果应当显示 Python 3.11、x86_64、PyTorch 2.2.2，并且 `MPS built` 和
`MPS available` 都是 `True`。如果代码停在某个断言，错误信息会直接指出需要
修正的项目。

In [1]:
import os
import platform
import sys

import numpy as np
import torch

print(f"Python executable : {os.path.basename(sys.executable)}")
print(f"Python version    : {platform.python_version()}")
print(f"Architecture      : {platform.machine()}")
print(f"NumPy             : {np.__version__}")
print(f"PyTorch           : {torch.__version__}")
print(f"MPS built         : {torch.backends.mps.is_built()}")
print(f"MPS available     : {torch.backends.mps.is_available()}")

assert platform.machine() == "x86_64", "这份教程只验证了 x86_64 环境"
assert platform.python_version_tuple()[:2] == ("3", "11"), "请改用 Python 3.11"
assert torch.__version__.startswith("2.2.2"), "请安装 PyTorch 2.2.2"
assert torch.backends.mps.is_built(), "当前 PyTorch 不包含 MPS 支持"
assert torch.backends.mps.is_available(), "系统没有提供可用的 MPS 设备"

fallback = os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK", "").lower()
assert fallback not in {"1", "true", "yes"}, (
    "请关闭 PYTORCH_ENABLE_MPS_FALLBACK，然后重新启动 JupyterLab"
)

print("\n环境检查通过，可以继续。")

Python executable : python
Python version    : 3.11.15
Architecture      : x86_64
NumPy             : 1.26.4
PyTorch           : 2.2.2
MPS built         : True
MPS available     : True

环境检查通过，可以继续。


## 第二步：确认 PyTorch 调用的是 RX 5700 XT

下面的代码会在 MPS 设备上创建张量并完成一次小计算。最重要的结果是：

```text
PASS ... device=mps:0
```

这台电脑没有核显，因此 `mps:0` 就是 RX 5700 XT。看到这行结果，说明计算通过
Metal 在显卡上执行，而不是落在 Ryzen CPU 上。

In [2]:
# 可以把这段最小验证代码复制到其他项目中
import torch

assert torch.backends.mps.is_available(), "MPS 不可用"

device = torch.device("mps")
x = torch.tensor([1.0, 2.0, 3.0], device=device)
y = x.square()
torch.mps.synchronize()

assert y.tolist() == [1.0, 4.0, 9.0]
print(f"PASS: {y} · device={y.device}")

PASS: tensor([1., 4., 9.], device='mps:0') · device=mps:0


## 第三步：跑一个简短性能测试

下面会进行 10 次 `4096 × 4096` 的 FP16 矩阵乘法，前两次只用于预热，通常
几秒钟就能完成。

这个成绩适合用来快速判断显卡加速是否正常，不是统一的显卡跑分。后台任务、显卡
频率和散热都会影响结果。运行时可以打开活动监视器、iStat Menus 或其他工具，
观察 GPU 占用和温度。

In [3]:
import time

matrix_size = 4096
matrix_a = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)
matrix_b = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)

# 先预热两次，避免第一次编译 Metal kernel 的时间影响成绩
torch.mps.synchronize()
for _ in range(2):
    result = matrix_a @ matrix_b
torch.mps.synchronize()

timed_iterations = 10
started = time.perf_counter()
for _ in range(timed_iterations):
    result = matrix_a @ matrix_b
torch.mps.synchronize()
elapsed = time.perf_counter() - started

operations = 2 * matrix_size ** 3 * timed_iterations
tflops = operations / elapsed / 1e12

print(f"Matrix       : {matrix_size} x {matrix_size}")
print(f"Device       : {result.device}")
print(f"Iterations   : {timed_iterations}")
print(f"Elapsed      : {elapsed:.4f} seconds")
print(f"Throughput   : {tflops:.2f} TFLOP/s")
print(f"Checksum     : {result[0, 0].item():.6g}")

assert str(result.device) == "mps:0"
assert tflops > 0

del matrix_a, matrix_b, result
torch.mps.empty_cache()

Matrix       : 4096 x 4096
Device       : mps:0
Iterations   : 10
Elapsed      : 0.2161 seconds
Throughput   : 6.36 TFLOP/s
Checksum     : -22.9531


## 本机实测结果

以下数据来自本 EFI 对应的 Ryzen 9 3900X + RX 5700 XT 电脑：

| 测试项目 | 本机结果 |
|---|---:|
| 最小 MPS 验证 | 通过，设备为 `mps:0` |
| Notebook 短测 | **6.36 TFLOP/s** |
| 五分钟压力测试时长 | 300.74 秒 |
| 完成的矩阵乘法 | 526 次 |
| 五分钟平均性能 | **6.49 TFLOP/s** |
| GPU 占用 | 约 95%–100% |
| 最高 GPU Die 温度 | 97°C |
| MPS / Driver 内存 | 约 0.84 / 1.13 GiB |

真正的通过条件是出现 `mps:0`。TFLOP/s 只代表这一种矩阵运算的速度，不能直接
当作模型训练性能。

持续压力测试能够让 RX 5700 XT 保持接近满载，说明 MPS 加速工作正常。测试中
观察到的 97°C 已经偏高，准备连续训练数小时之前应先改善散热。

## 在自己的项目中使用

模型和输入数据必须放到同一个设备：

```python
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = model.to(device)
inputs = inputs.to(device)
outputs = model(inputs)
```

如果 PyTorch 提示模型与输入不在同一设备，可以分别检查：

```python
print(next(model.parameters()).device)
print(inputs.device)
```

实际使用时还需要注意：

- RX 5700 XT 有 8 GB 显存。显存不足时先减小 `batch_size` 或输入尺寸。
- 建议先用 `float32` 跑通模型，再尝试 `float16`。
- PyTorch 2.2.2 的 AMD MPS 不支持 `bfloat16`。
- 少数算子没有 MPS 实现。设置 `PYTORCH_ENABLE_MPS_FALLBACK=1` 可以让它们
  暂时回到 CPU 执行，但速度可能明显下降。
- 检查显卡加速时应关闭 fallback，并确认张量设备显示为 `mps:0`。

到这里，这套环境就可以用于普通 PyTorch 项目了。